# 基于BERT的中文文本分类 - 外卖评论二分类（离线模式）

## 项目说明
本项目使用本地 bert-base-chinese 模型对 waimai_1k.csv 数据进行二分类训练，实现外卖评论的情感分类任务。

## 模型文件夹位置
模型保存在：`e:\BertTunning\bert-base-chinese\`

### 需要下载的文件：
- config.json（模型配置文件）
- pytorch_model.bin（预训练模型权重）
- vocab.txt（词汇表）
- tokenizer_config.json（分词器配置）

### 下载地址：
https://huggingface.co/bert-base-chinese/tree/main

---

## 项目结构（改进版）
1. **环境配置**：导入必要的库和设置路径
2. **配置参数**：设置超参数和模型保存路径
3. **随机种子设置**：确保结果可复现
4. **数据准备**：封装的数据加载函数
5. **模型定义**：定义数据集类和BERT分类模型
6. **训练与评估**：包含最佳模型保存的完整训练流程

In [6]:
"""
==========================================
第一部分：导入必要的库
==========================================
本部分导入项目所需的所有Python库和模块
"""

# PyTorch相关库
import torch                    # PyTorch核心库，用于张量操作和自动求导
import torch.nn as nn           # 神经网络模块，用于定义模型层
from torch.utils.data import DataLoader, Dataset  # 数据加载器，用于批量处理数据

# 数据处理库
import pandas as pd             # 用于读取和处理CSV数据
import numpy as np              # 用于数值计算和数组操作

# 系统库
import os                       # 用于文件路径操作和检查文件是否存在
import random                   # 用于设置随机种子

# Transformers库（Hugging Face）
from transformers import BertTokenizer, BertModel  # BERT分词器和模型

# 优化器和工具
from torch.optim import Adam    # Adam优化器，用于模型参数更新
from tqdm import tqdm           # 进度条显示，用于训练过程可视化
from torch.optim import AdamW  # 使用AdamW代替Adam

# 数据分割工具
from sklearn.model_selection import train_test_split  # 用于划分训练集、验证集和测试集

print("✅ 所有库导入完成")

✅ 所有库导入完成


In [3]:
"""
==========================================
第二部分：配置参数
==========================================
设置模型路径、超参数等配置信息
"""

# ========== 模型路径配置 ==========
# 设置本地 BERT 模型路径（离线模式）
# 注意：需要提前下载 bert-base-chinese 模型到该路径
BERT_MODEL_PATH = r'../bert-base-chinese'
DATA_DIR = '../waimai.csv'

# 检查模型文件夹是否存在
if not os.path.exists(BERT_MODEL_PATH):
    print(f"⚠️  警告：模型文件夹不存在: {BERT_MODEL_PATH}")
else:
    print(f"✅ 模型路径检查通过: {BERT_MODEL_PATH}")

# ========== 超参数配置 ==========
# 训练相关参数
EPOCHS = 3              # 训练轮数
LEARNING_RATE = 2e-5    # 学习率（建议范围：1e-5 到 5e-5）
BATCH_SIZE = 16         # 批次大小（根据GPU内存调整，CPU建议16-32）

# 模型相关参数
MAX_LENGTH = 512        # 文本最大长度（BERT最大支持512）
DROPOUT = 0.5           # Dropout比率，用于防止过拟合
NUM_CLASSES = 2         # 分类类别数（二分类：正面/负面）

# 数据分割比例
TRAIN_RATIO = 0.7       # 训练集比例
VAL_RATIO = 0.15        # 验证集比例
TEST_RATIO = 0.15       # 测试集比例

# ========== 随机种子和模型保存配置 ==========
RANDOM_SEED = 42        # 随机种子，确保结果可复现
SAVE_PATH = './bert_checkpoint'  # 模型保存路径

print(f"✅ 配置参数设置完成")
print(f"   - 训练轮数: {EPOCHS}")
print(f"   - 学习率: {LEARNING_RATE}")
print(f"   - 批次大小: {BATCH_SIZE}")
print(f"   - 随机种子: {RANDOM_SEED}")
print(f"   - 模型保存路径: {SAVE_PATH}")

✅ 模型路径检查通过: ../bert-base-chinese
✅ 配置参数设置完成
   - 训练轮数: 3
   - 学习率: 1e-05
   - 批次大小: 16
   - 随机种子: 42
   - 模型保存路径: ./bert_checkpoint


In [4]:
"""
==========================================
第三部分：设置随机种子
==========================================
确保实验结果可复现，所有随机操作都会使用相同的种子
"""

def setup_seed(seed):
    """
    设置随机种子，确保实验结果可复现
    
    参数:
        seed: 随机种子值
    """
    torch.manual_seed(seed)              # 设置PyTorch的随机种子
    torch.cuda.manual_seed_all(seed)     # 设置所有GPU的随机种子
    np.random.seed(seed)                 # 设置NumPy的随机种子
    random.seed(seed)                    # 设置Python内置random的随机种子
    torch.backends.cudnn.deterministic = True  # 确保CUDA操作可复现

# 应用随机种子
setup_seed(RANDOM_SEED)

print("✅ 随机种子设置完成，实验结果可复现")

✅ 随机种子设置完成，实验结果可复现


In [5]:
"""
==========================================
第四部分：初始化分词器
==========================================
加载BERT分词器，用于将中文文本转换为模型可理解的token序列
"""

# 从本地路径加载BERT分词器
# 分词器会将文本转换为：
# 1. input_ids: token的ID序列
# 2. attention_mask: 标记哪些位置是真实token（1）还是padding（0）
# 3. token_type_ids: 用于区分不同句子（单句分类任务中通常不需要）
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_PATH)

print("✅ 分词器加载完成")
print(f"   - 词汇表大小: {len(tokenizer.vocab)}")
print(f"   - 特殊token示例: [CLS]={tokenizer.cls_token}, [SEP]={tokenizer.sep_token}, [PAD]={tokenizer.pad_token}")

✅ 分词器加载完成
   - 词汇表大小: 21128
   - 特殊token示例: [CLS]=[CLS], [SEP]=[SEP], [PAD]=[PAD]


In [ ]:
"""
==========================================
第五部分：定义数据集类
==========================================
自定义Dataset类，用于将原始数据转换为模型可用的格式
"""

class TextDataset(Dataset):
    """
    文本分类数据集类
    
    功能：
    1. 将原始文本数据转换为BERT可接受的格式
    2. 对文本进行分词、padding和截断处理
    3. 返回处理后的文本和对应的标签
    """
    
    def __init__(self, df):
        """
        初始化数据集
        
        参数:
            df: pandas DataFrame，包含 'review' 和 'label' 列
        """
        # 将标签转换为numpy数组，避免pandas索引问题
        self.labels = df['label'].astype(int).values
        
        # 对每个文本进行分词处理
        # padding='max_length': 将文本填充到最大长度512
        # max_length=512: BERT支持的最大序列长度
        # truncation=True: 如果文本超过512，则截断
        # return_tensors="pt": 返回PyTorch张量格式
        #输出的texts中包含
        # - input_ids：token ID序列（就是你问的input_ids） 这个其实是输入review被分词+编码后的结果
        # - attention_mask 这个是因为长度固定为512，没达到512的文本token默认补0。使用这个mask标记一下补充0的位置，好让模型推理时自动忽略掉。
        self.texts = [
            tokenizer(
                text, 
                padding='max_length', 
                max_length=MAX_LENGTH, 
                truncation=True,
                return_tensors="pt"
            ) 
            for text in df['review']
        ]
        
    def __len__(self):
        """返回数据集大小"""
        return len(self.labels)
    #需要输出每一行数据的[input_ids,attention_mask],label
    def __getitem__(self, idx):
        """
        获取单个数据样本
        
        参数:
            idx: 数据索引
            
        返回:
            batch_texts: 处理后的文本（包含input_ids和attention_mask）
            batch_y: 对应的标签
        """
        batch_texts = self.get_batch_texts(idx)
        batch_y = self.get_batch_labels(idx)
        return batch_texts, batch_y
    
    def classes(self):
        """返回所有标签"""
        return self.labels
    
    def get_batch_labels(self, idx):
        """
        获取标签并转换为Long类型张量
        CrossEntropyLoss需要Long类型的标签
        """
        return torch.tensor(self.labels[idx], dtype=torch.long)

    def get_batch_texts(self, idx):
        """获取处理后的文本数据"""
        return self.texts[idx]

print("✅ 数据集类定义完成")

In [ ]:
"""
==========================================
第六部分：数据加载函数
==========================================
封装数据加载和分割逻辑，支持按模式返回不同的数据集
"""

def GenerateData(mode='train'):
    """
    加载和预处理数据，并返回指定模式的数据集
    
    参数:
        mode: 数据模式，可选 'train'、'val' 或 'test'
    
    返回:
        TextDataset: 处理后的数据集对象
    """
    # 读取CSV数据文件
    # 数据文件应包含 'review'（评论文本）和 'label'（标签：0或1）两列
    data_path = DATA_DIR
    df = pd.read_csv(data_path)
    
    # 查看数据基本信息（仅在第一次调用时显示）
    if mode == 'train':
        print("=" * 50)
        print("数据基本信息")
        print("=" * 50)
        print(f"数据总量: {len(df)}")
        print(f"\n标签分布:")
        print(df['label'].value_counts())
        print(f"\n数据示例（前3条）:")
        print(df.head(3))
    
    # ========== 数据分割 ==========
    # 使用分层抽样（stratify）确保训练集、验证集和测试集的标签分布一致
    # 第一次分割：70%训练集，30%临时集
    train_df, temp_df = train_test_split(
        df,
        test_size=0.3,
        stratify=df['label'],  # 分层抽样，保持标签比例
        random_state=RANDOM_SEED  # 使用配置的随机种子，确保结果可复现
    )
    
    # 第二次分割：将30%的临时集分为15%验证集和15%测试集
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df['label'],
        random_state=RANDOM_SEED
    )
    
    # 根据模式返回对应的数据集
    if mode == 'train':
        print("\n" + "=" * 50)
        print("数据分割结果")
        print("=" * 50)
        print(f"训练集大小: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
        print(f"验证集大小: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
        print(f"测试集大小: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
        print("\n✅ 数据加载和预处理完成")
        return TextDataset(train_df)
    elif mode == 'val':
        return TextDataset(val_df)
    elif mode == 'test':
        return TextDataset(test_df)
    else:
        raise ValueError(f"不支持的模式: {mode}，请使用 'train'、'val' 或 'test'")

print("✅ 数据加载函数定义完成")

In [ ]:
"""
==========================================
第七部分：定义BERT分类模型
==========================================
构建基于BERT的文本分类模型
"""

class BertClassifier(nn.Module):
    """
    BERT文本分类模型
    
    模型结构：
    1. BERT编码器：提取文本特征
    2. Dropout层：防止过拟合
    3. 全连接层：将768维特征映射到2个类别
    4. ReLU激活函数：增加非线性
    """
    
    def __init__(self, dropout=DROPOUT, model_path=BERT_MODEL_PATH,freeze_bert=False):
        """
        初始化模型
        
        参数:
            dropout: Dropout比率，默认0.5
            model_path: BERT模型路径
        """
        super(BertClassifier, self).__init__()
        
        # 加载预训练的BERT模型（不包含分类头）
        # 使用本地模型路径，支持离线运行
        self.bert = BertModel.from_pretrained(model_path)

        # 如果freeze_bert为True，则冻结BERT参数
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
            print("bert主干参数已经被冻结")
        
        # Dropout层：随机丢弃50%的神经元，防止过拟合
        self.dropout = nn.Dropout(dropout)
        
        # 全连接层：BERT输出维度是768，映射到2个类别（正面/负面）
        self.linear = nn.Linear(768, NUM_CLASSES)
        
        # ReLU激活函数：增加模型的非线性表达能力
        # self.relu = nn.ReLU()

    def forward(self, input_id, mask):
        """
        前向传播
        
        参数:
            input_id: token ID序列，形状为 [batch_size, seq_length]
            mask: attention mask，形状为 [batch_size, seq_length]
            
        返回:
            final_layer: 分类结果，形状为 [batch_size, num_classes]
        """
        # 通过BERT模型获取文本表示
        # return_dict=False: 返回元组而不是字典
        # pooled_output: [CLS] token的表示，包含了整个句子的语义信息
        _, pooled_output = self.bert(
            input_ids=input_id, 
            attention_mask=mask,
            return_dict=False
        )
        
        # 应用Dropout
        dropout_output = self.dropout(pooled_output)
        
        # 通过全连接层得到分类logits
        linear_output = self.linear(dropout_output)
        
        # 应用ReLU激活函数 好像不能直接接relu
        # final_layer = self.relu(linear_output)
        
        return linear_output

print("✅ BERT分类模型类定义完成")

In [ ]:
"""
==========================================
第八部分：初始化模型
==========================================
创建BERT分类模型实例
"""

# 使用本地模型路径初始化分类器
# 模型会自动加载预训练的BERT权重
model = BertClassifier(model_path=BERT_MODEL_PATH,freeze_bert=True)

# 输出模型信息
print("✅ BERT 模型初始化完成")
print(f"   - 模型参数数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"   - 可训练参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 注意：如果看到关于未使用权重的警告，这是正常的
# 因为我们只使用BERT的编码器部分，而不需要预训练时的分类头

In [ ]:
"""
==========================================
第九部分：准备训练数据
==========================================
使用GenerateData函数加载训练集和验证集，并创建DataLoader
"""

# ========== 数据准备 ==========
# 使用GenerateData函数加载数据集
train_dataset = GenerateData(mode='train')
val_dataset = GenerateData(mode='val')

# 创建DataLoader，用于批量加载数据
# shuffle=True: 训练时打乱数据顺序，提高模型泛化能力
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_dataloader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE
)

print("✅ 训练数据准备完成")
print(f"   - 训练批次数: {len(train_dataloader)}")
print(f"   - 验证批次数: {len(val_dataloader)}")

In [ ]:
"""
==========================================
第十部分：配置训练环境
==========================================
设置设备、损失函数和优化器
"""

# ========== 设备配置 ==========
# 检查是否有可用的GPU，如果有则使用GPU加速训练
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"使用设备: {device}")

# ========== 损失函数和优化器 ==========
# CrossEntropyLoss: 多分类交叉熵损失函数
criterion = nn.CrossEntropyLoss()

# Adam优化器：自适应学习率的优化算法
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,  # 添加L2正则化,
    eps=1e-8
)

# 将模型和损失函数移到指定设备（CPU或GPU）
if use_cuda:
    model = model.cuda()
    criterion = criterion.cuda()

print("✅ 训练环境配置完成")

In [ ]:
"""
==========================================
第十一部分：模型保存函数
==========================================
定义模型保存函数，用于保存训练过程中的最佳模型和最后模型
"""

def save_model(model, save_name):
    """
    保存模型状态字典
    
    参数:
        model: 要保存的模型对象
        save_name: 保存的文件名（如 'best.pt' 或 'last.pt'）
    """
    # 如果保存路径不存在，则创建
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
        print(f"✅ 创建模型保存目录: {SAVE_PATH}")
    
    # 保存模型状态字典
    save_file = os.path.join(SAVE_PATH, save_name)
    torch.save(model.state_dict(), save_file)
    print(f"✅ 模型已保存: {save_file}")

print("✅ 模型保存函数定义完成")

In [ ]:
"""
==========================================
第十二部分：训练模型
==========================================
在训练集上训练模型，并在验证集上评估性能
包含最佳模型保存机制，自动保存验证集上表现最好的模型
"""

# ========== 训练循环 ==========
print("\n开始训练...")

# 初始化最佳验证准确率，用于跟踪和保存最佳模型
best_dev_acc = 0

for epoch_num in range(EPOCHS):
    # ========== 训练阶段 ==========
    model.train()  # 设置为训练模式，启用Dropout等训练特性
    
    # 初始化累计指标
    total_acc_train = 0
    total_loss_train = 0
    
    # 遍历训练数据批次
    for train_input, train_label in tqdm(
        train_dataloader, 
        desc=f"Epoch {epoch_num + 1}/{EPOCHS}"
    ):
        # 将数据移到指定设备
        train_label = train_label.to(device)
        mask = train_input['attention_mask'].to(device)
        input_id = train_input['input_ids'].squeeze(1).to(device)
        
        # 前向传播：通过模型得到预测结果
        output = model(input_id, mask)
        
        # 计算损失
        batch_loss = criterion(output, train_label)
        total_loss_train += batch_loss.item()
        
        # 计算准确率：预测类别与真实标签一致的数量
        acc = (output.argmax(dim=1) == train_label).sum().item()
        total_acc_train += acc
        
        # 反向传播和参数更新
        model.zero_grad()      # 清零梯度 grad相关的就是梯度
        batch_loss.backward()  # 反向传播计算梯度
        # 这里loss函数是自动与整个模型相连接的，即使在模型外定义loss函数，这是torch自动完成的。
        optimizer.step()       # 更新模型参数
    
    # ========== 验证阶段 ==========
    model.eval()  # 设置为评估模式，禁用Dropout等训练特性
    
    # 初始化累计指标
    total_acc_val = 0
    total_loss_val = 0
    
    # 验证时不需要计算梯度，节省内存和计算资源
    with torch.no_grad():
        for val_input, val_label in val_dataloader:
            # 将数据移到指定设备
            val_label = val_label.to(device)
            mask = val_input['attention_mask'].to(device)
            input_id = val_input['input_ids'].squeeze(1).to(device)
            
            # 前向传播
            output = model(input_id, mask)
            
            # 计算损失和准确率
            batch_loss = criterion(output, val_label)
            total_loss_val += batch_loss.item()
            
            acc = (output.argmax(dim=1) == val_label).sum().item()
            total_acc_val += acc
    
    # ========== 输出训练结果 ==========
    val_accuracy = total_acc_val / len(val_dataset)
    print(
        f'''Epochs: {epoch_num + 1} 
          | Train Loss: {total_loss_train / len(train_dataset): .3f} 
          | Train Accuracy: {total_acc_train / len(train_dataset): .3f} 
          | Val Loss: {total_loss_val / len(val_dataset): .3f} 
          | Val Accuracy: {val_accuracy: .3f}'''
    )
    
    # ========== 保存最佳模型 ==========
    # 如果当前验证准确率超过历史最佳，则保存模型
    if val_accuracy > best_dev_acc:
        best_dev_acc = val_accuracy
        save_model(model, 'best.pt')
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")

# 保存最后一个epoch的模型
save_model(model, 'last.pt')

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best.pt')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last.pt')}")

In [ ]:
"""
==========================================
第十三部分：在测试集上评估模型
==========================================
使用训练好的模型在测试集上进行最终评估
注意：测试集在整个训练过程中都没有被使用，用于评估模型的真实泛化能力
"""

# ========== 准备测试数据 ==========
# 使用GenerateData函数加载测试集
test_dataset = GenerateData(mode='test')
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# ========== 评估模型 ==========
# 设置为评估模式
model.eval()

# 初始化累计指标
total_acc_test = 0
total_loss_test = 0

# 在测试集上评估（不需要计算梯度）
with torch.no_grad():
    for test_input, test_label in tqdm(test_dataloader, desc="测试中"):
        # 将数据移到指定设备
        test_label = test_label.to(device)
        mask = test_input['attention_mask'].to(device)
        input_id = test_input['input_ids'].squeeze(1).to(device)
        
        # 前向传播
        output = model(input_id, mask)
        
        # 计算损失和准确率
        batch_loss = criterion(output, test_label)
        total_loss_test += batch_loss.item()
        
        acc = (output.argmax(dim=1) == test_label).sum().item()
        total_acc_test += acc

# ========== 输出评估结果 ==========
# 计算平均损失和准确率
avg_loss = total_loss_test / len(test_dataset)
avg_accuracy = total_acc_test / len(test_dataset)

print(f"\n测试集评估结果:")
print(f"  - Loss: {avg_loss: .3f}")
print(f"  - Accuracy: {avg_accuracy: .3f}")

print(f"\n🎉 最终测试准确率: {avg_accuracy:.3f}")

## 训练完成！

### 模型性能总结
- **训练集**：用于训练模型参数
- **验证集**：用于调整超参数和监控训练过程，并选择最佳模型
- **测试集**：用于最终评估模型泛化能力

### 改进特性
1. ✅ **可复现性**：通过`setup_seed()`函数确保实验结果可复现
2. ✅ **代码复用**：通过`GenerateData()`函数封装数据加载逻辑
3. ✅ **模型管理**：自动保存最佳模型（best.pt）和最后模型（last.pt）
4. ✅ **完整流程**：训练-验证-测试的完整流程
5. ✅ **详细注释**：保留详细注释和模块化结构，便于理解和维护

### 模型保存位置
- 最佳模型：`./bert_checkpoint/best.pt`
- 最后模型：`./bert_checkpoint/last.pt`

### 加载保存的模型
```python
# 加载最佳模型
model = BertClassifier()
model.load_state_dict(torch.load('./bert_checkpoint/best.pt'))
model.eval()
```

### 后续优化建议
1. **调整超参数**：尝试不同的学习率、批次大小、训练轮数
2. **数据增强**：增加训练数据量或使用数据增强技术
3. **模型微调**：尝试不同的dropout率或添加更多全连接层
4. **早停机制**：当验证集准确率不再提升时提前停止训练
5. **学习率调度**：使用学习率衰减策略优化训练过程

In [ ]:
"""
==========================================
第十四部分：文本预测函数
==========================================
使用训练好的模型对新的中文文本进行预测
"""

def predict_text(text, model_path='best'):
    """
    预测单个中文文本的情感类别
    
    参数:
        text: 要预测的中文文本（字符串）
        model_path: 模型文件路径，'best' 表示使用 best.pt，'last' 表示使用 last.pt
                   或者直接指定完整路径
    
    返回:
        prediction: 预测的类别（0=负面，1=正面）
        probability: 预测的概率分布
        confidence: 预测的置信度（最大概率值）
    """
    # 加载模型（如果还没有加载或需要重新加载）
    # 注意：如果模型已经在内存中，可以跳过这一步
    # if not hasattr(predict_text, 'loaded_model') or predict_text.model_path != model_path:
    #     # 确定模型文件路径
    #     if model_path == 'best':
    #         model_file = os.path.join(SAVE_PATH, 'best.pt')
    #     elif model_path == 'last':
    #         model_file = os.path.join(SAVE_PATH, 'last.pt')
    #     else:
    #         model_file = model_path
    #
    #     # 检查模型文件是否存在
    #     if not os.path.exists(model_file):
    #         raise FileNotFoundError(f"模型文件不存在: {model_file}")
    #
    #     # 创建模型实例并加载权重
    #     predict_text.loaded_model = BertClassifier(model_path=BERT_MODEL_PATH)
    #     predict_text.loaded_model.load_state_dict(torch.load(model_file, map_location=device))
    #     predict_text.loaded_model.to(device)
    #     predict_text.loaded_model.eval()  # 设置为评估模式
    #     predict_text.model_path = model_path
    #     print(f"✅ 模型已加载: {model_file}")
    #
    # model = predict_text.loaded_model
    
    # 对文本进行预处理（分词、编码）
    encoded_text = tokenizer(
        text,
        padding='max_length',
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors="pt"
    )
    
    # 将数据移到指定设备
    input_id = encoded_text['input_ids'].squeeze(1).to(device)
    mask = encoded_text['attention_mask'].to(device)
    
    # 进行预测（不需要计算梯度）
    with torch.no_grad():
        output = model(input_id, mask)
        
        # 使用softmax将logits转换为概率
        probabilities = torch.nn.functional.softmax(output, dim=1)
        
        # 获取预测类别（概率最大的类别）
        prediction = output.argmax(dim=1).item()
        
        # 获取置信度（最大概率值）
        confidence = probabilities[0][prediction].item()
        
        # 获取所有类别的概率
        prob_dist = probabilities[0].cpu().numpy()
    
    return prediction, prob_dist, confidence

# 初始化函数属性（用于缓存模型）
predict_text.loaded_model = None
predict_text.model_path = None

# ========== 示例预测 ==========
print("=" * 50)
print("文本预测示例")
print("=" * 50)

# 示例文本1：正面评论
text1 = "这家店的外卖很好吃，配送也很快，包装很精美，下次还会再点！"
prediction1, prob1, confidence1 = predict_text(text1)

print(f"\n📝 文本1: {text1}")
print(f"   预测类别: {'正面' if prediction1 == 1 else '负面'} (类别编号: {prediction1})")
print(f"   置信度: {confidence1:.4f}")
print(f"   概率分布: 负面={prob1[0]:.4f}, 正面={prob1[1]:.4f}")

# 示例文本2：负面评论
text2 = "送餐太慢了，等了两个小时才到，而且食物都凉了，非常不满意。"
prediction2, prob2, confidence2 = predict_text(text2)

print(f"\n📝 文本2: {text2}")
print(f"   预测类别: {'正面' if prediction2 == 1 else '负面'} (类别编号: {prediction2})")
print(f"   置信度: {confidence2:.4f}")
print(f"   概率分布: 负面={prob2[0]:.4f}, 正面={prob2[1]:.4f}")

# 示例文本3：中性/模糊评论
text3 = "还可以吧，一般般，没什么特别的。"
prediction3, prob3, confidence3 = predict_text(text3)

print(f"\n📝 文本3: {text3}")
print(f"   预测类别: {'正面' if prediction3 == 1 else '负面'} (类别编号: {prediction3})")
print(f"   置信度: {confidence3:.4f}")
print(f"   概率分布: 负面={prob3[0]:.4f}, 正面={prob3[1]:.4f}")

print("\n" + "=" * 50)
print("✅ 预测功能已就绪！")
print("=" * 50)
print("\n💡 使用方法:")
print("   prediction, probability, confidence = predict_text('你的中文文本')")
print("   print(f'预测结果: {\"正面\" if prediction == 1 else \"负面\"}')")
print("   print(f'置信度: {confidence:.2%}')")